In [10]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os

In [41]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os

# ────────────────────────────────────────────────
# Load TerraClimate dataset
# ────────────────────────────────────────────────
def load_terraclimate_dataset():
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )

    return ds

# ────────────────────────────────────────────────
# Filter dataset to time range (now 2000-2021)
# ────────────────────────────────────────────────
def filter_dataset(ds):
    ds_2000_2021 = ds.sel(time=slice("2000-01-01", "2021-12-31"))
    return ds_2000_2021

# ────────────────────────────────────────────────
# Assign nearest climate values using cKDTree (modified for full series)
# ────────────────────────────────────────────────
def assign_nearest_climate(sa_df, climate_df, var_name):
    # Map nearest climate variable values to a new DataFrame
    # containing only the specified variable column. For full series per point.
    
    sa_coords = np.radians(sa_df[["Latitude", "Longitude"]].values)
    climate_coords = np.radians(climate_df[["lat", "lon"]].values)
    
    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)
    
    nearest_points = climate_df.iloc[idx].reset_index(drop=True)
    sa_df = sa_df.reset_index(drop=True)
    sa_df["nearest_lat"] = nearest_points["lat"]
    sa_df["nearest_lon"] = nearest_points["lon"]
    
    climate_values = []
    climate_lats = []
    climate_lons = []
    
    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        nearest_lat = sa_df.loc[i, "nearest_lat"]
        nearest_lon = sa_df.loc[i, "nearest_lon"]
        
        subset = climate_df[
            (climate_df["lat"] == nearest_lat) &
            (climate_df["lon"] == nearest_lon)
        ]
        
        if subset.empty:
            continue  # skip if no match
        
        # Append full series for this point
        for _, row in subset.iterrows():
            climate_lats.append(sa_df.loc[i, "Latitude"])
            climate_lons.append(sa_df.loc[i, "Longitude"])
            climate_values.append(row[var_name])
    
    output_df = pd.DataFrame({
        "Latitude": climate_lats,
        "Longitude": climate_lons,
        "PET": climate_values
    })
    
    return output_df

# ────────────────────────────────────────────────
# Main extraction process
# ────────────────────────────────────────────────

# Load full dataset
ds = load_terraclimate_dataset()

# Filter to 2000-2021
ds_filtered = filter_dataset(ds)

# Extract PET variable as DataFrame (for easier processing)
# This creates a flat DF with all lat/lon/time combinations
print("Converting PET to DataFrame...")
pet_df = ds_filtered["pet"].stack(z=("lat", "lon")).to_dataframe().reset_index()
pet_df.rename(columns={"pet": "PET"}, inplace=True)
pet_df.drop(columns=["z"], inplace=True, errors="ignore")  # clean up

# Load your training CSV for lat/lon
water_df = pd.read_csv("water_quality_training_dataset.csv")

# Assign nearest PET (full series for each point)
output_df = assign_nearest_climate(water_df, pet_df, "PET")

# Save CSV
output_df.to_csv("pet_2000_2021.csv", index=False)

print("Extraction complete. Saved to pet_2000_2021.csv")

# Load and display the dataset
loaded_df = pd.read_csv("pet_2000_2021.csv")
print("\nFull extracted dataset table:")
print(loaded_df.to_string(index=False))

Converting PET to DataFrame...


: 

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr

import pystac_client
import planetary_computer as pc

from tqdm import tqdm

def load_terraclimate_dataset():
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )

    return ds

def filter_dataset(ds):
    ds_2000_2005 = ds.sel(time=slice("2000-01-01", "2005-12-31"))
    return ds_2000_2005

# Load full dataset
ds = load_terraclimate_dataset()

# Filter to 2000-2005
ds_filtered = filter_dataset(ds)

# Load your training CSV for lat/lon
water_df = pd.read_csv("water_quality_training_dataset.csv")
unique_points = water_df[["Latitude", "Longitude"]].drop_duplicates().reset_index(drop=True)

# Extract PET time series for each unique point (memory efficient - no full DF)
results = []
pet_da = ds_filtered["pet"]

with tqdm(total=len(unique_points), desc="Extracting PET 2000-2005") as pbar:
    for _, row in unique_points.iterrows():
        lat = row["Latitude"]
        lon = row["Longitude"]
        
        # Nearest cell + full time series for 2000-2005
        try:
            ts = pet_da.sel(lat=lat, lon=lon, method="nearest")
            df_ts = ts.to_dataframe().reset_index(drop=True)
            df_ts = df_ts.rename(columns={"pet": "PET"})
            df_ts["Latitude"] = lat
            df_ts["Longitude"] = lon
            df_ts = df_ts[["Latitude", "Longitude", "PET"]]  # Only lat, lon, PET
            results.append(df_ts)
        except Exception as e:
            print(f"Skip lat={lat:.5f} lon={lon:.5f}: {e}")
        
        pbar.update(1)

# Combine and save
if results:
    output_df = pd.concat(results, ignore_index=True)
    output_df.to_csv("pet_2000_2005.csv", index=False)
    print("Extraction complete. Saved to pet_2000_2005.csv")

    # Load and display full table
    loaded_df = pd.read_csv("pet_2000_2005.csv")
    print("\nFull extracted dataset table:")
    print(loaded_df.to_string(index=False))
else:
    print("No data extracted.")

print("Finished.")

Extracting PET 2000-2005:   0%|          | 0/162 [00:26<?, ?it/s]


KeyboardInterrupt: 